# 🐹 倉鼠辨識模型訓練（YOLO11n → 樹莓派 NCNN）

1. 上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**
2. 由上往下一格一格按 ▶ 執行
3. 最後會下載 `hamster_ncnn_model.zip`，交給樹莓派使用


In [ ]:
# 1. 安裝 Ultralytics，確認有 GPU
!pip -q install ultralytics
import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '沒有 GPU（會很慢，請改成 T4 GPU）')

In [ ]:
# 2. 上傳標記好的 hamster_dataset.zip（裡面是照片 + 同名 .txt 標記 + classes.txt）
from google.colab import files
import zipfile, pathlib, shutil
up = files.upload()
name = next(iter(up))
shutil.rmtree('/content/raw', ignore_errors=True)
zipfile.ZipFile(name).extractall('/content/raw')
imgs = [p for p in pathlib.Path('/content/raw').rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
labeled = [p for p in imgs if p.with_suffix('.txt').exists() and p.with_suffix('.txt').read_text().strip()]
print(f'照片 {len(imgs)} 張，有框的 {len(labeled)} 張，其他 {len(imgs) - len(labeled)} 張當作背景（沒有倉鼠）')
# class 0 = hamster (whole body), class 1 = head
cls = lambda p: [l.split()[0] for l in p.with_suffix('.txt').read_text().splitlines() if l.split()]
n_ham = sum('0' in cls(p) for p in labeled); n_head = sum('1' in cls(p) for p in labeled)
print(f'有 hamster 框 {n_ham} 張，有 head 框 {n_head} 張')
START = next(pathlib.Path('/content/raw').rglob('best.pt'), None)   # the model in use (packed by the Pi): continue from it
from ultralytics import YOLO
BASE = YOLO(str(START)) if START else YOLO('yolo11n.pt')
IMGSZ = int((getattr(BASE, 'ckpt', None) or {}).get('train_args', {}).get('imgsz', 320) or 320) if START else 320
print(f'從現在的模型接著訓練（輸入 {IMGSZ}）' if START else '從 YOLO 預訓練模型開始（zip 裡沒有 best.pt）')
TRAIN_HEAD = n_head >= 50 or (START is not None and len(BASE.names) >= 2)   # too few heads: train hamster only
if not TRAIN_HEAD: print('head 框少於 50 張：這次只訓練 hamster（整隻），之後標滿再訓練頭')
elif n_head < 0.6 * n_ham: print('⚠ 很多有倉鼠的照片沒有框頭。看得到頭的照片都要框 head，否則模型會學到「這不是頭」')

In [ ]:
# 3. 分成訓練 85% / 驗證 15%，產生 data.yaml
import random
random.seed(0)
random.shuffle(imgs)
n_val = max(1, int(len(imgs) * 0.15))
root = pathlib.Path('/content/ds')
shutil.rmtree(root, ignore_errors=True)
for split, items in (('val', imgs[:n_val]), ('train', imgs[n_val:])):
    (root / 'images' / split).mkdir(parents=True)
    (root / 'labels' / split).mkdir(parents=True)
    for p in items:
        shutil.copy(p, root / 'images' / split / p.name)
        txt = p.with_suffix('.txt')
        # hamster (0) and head (1) boxes; missing txt = background image (empty label file)
        keep = ('0', '1') if TRAIN_HEAD else ('0',)
        lines = [l for l in (txt.read_text().splitlines() if txt.exists() else []) if l.split() and l.split()[0] in keep]
        (root / 'labels' / split / (p.stem + '.txt')).write_text('\n'.join(lines) + ('\n' if lines else ''))
names = '  0: hamster\n  1: head\n' if TRAIN_HEAD else '  0: hamster\n'
(root / 'data.yaml').write_text(f'path: {root}\ntrain: images/train\nval: images/val\nnames:\n{names}')
print('train', len(imgs) - n_val, '| val', n_val)

In [ ]:
# 4. 訓練（T4 GPU 約 10~30 分鐘）。imgsz=320 是為了讓樹莓派跑得快
from ultralytics import YOLO
model = BASE   # the model in use (best.pt in the zip) or the YOLO pre-trained one
model.train(data='/content/ds/data.yaml', imgsz=IMGSZ, epochs=60 if START else 120, patience=25, batch=32,
            project='/content/runs', name='hamster', exist_ok=True,
            fliplr=0.5, flipud=0.5, degrees=10, mosaic=1.0)   # top-down camera: flips/rotations are realistic

In [ ]:
# 5. 看成績：mAP50 越接近 1 越好（0.8 以上就很好用）
best = YOLO('/content/runs/hamster/weights/best.pt')
m = best.val(data='/content/ds/data.yaml', imgsz=IMGSZ)
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}   precision = {m.box.mp:.3f}   recall = {m.box.mr:.3f}')
from IPython.display import Image, display
for f in ('val_batch0_pred.jpg', 'confusion_matrix.png'):
    p = pathlib.Path('/content/runs/hamster') / f
    if p.exists(): display(Image(str(p), width=700))

In [ ]:
# 6. 轉成樹莓派用的 NCNN 格式並下載
out = best.export(format='ncnn', imgsz=IMGSZ)
src = pathlib.Path(out)
dst = pathlib.Path('/content/hamster_ncnn_model')
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(src, dst)
shutil.copy('/content/runs/hamster/weights/best.pt', dst / 'best.pt')   # keep the original for later fine-tuning
shutil.make_archive('/content/hamster_ncnn_model', 'zip', '/content', 'hamster_ncnn_model')
files.download('/content/hamster_ncnn_model.zip')

## 7. 手動跑分（官方工具，給比賽報告用）
不用訓練也能單獨跑：先執行第 1 格裝套件，再執行這一格。
上傳網站「跑分 → 資料 → 下載跑分照片」的 `hamster_bench.zip`；如果這次沒有在這個筆記本訓練，再上傳要測的模型 `best.pt`（網站「管理模型 → 下載」的 zip 裡有）。
會用 Ultralytics 官方 `val` 算各類別的 AP50、AP50-95 和固定信心門檻下的 Precision、Recall、F1，並畫出 PR 曲線和混淆矩陣。
接著把**同一個模型**匯出成 NCNN（樹莓派用的格式），在**同一組跑分照片**上再測一次，列出轉換前後的差異。這個差距是在驗證「匯出／轉換」造成的差異，不能說成是「因為換到樹莓派」。
只有全身一個類別的舊模型，請只比較「hamster」那一列；頭部另外看。

In [ ]:
# 7. 手動跑分：Ultralytics 官方 val + 轉換前後（best.pt vs NCNN）差異
from google.colab import files
import zipfile, pathlib, shutil, yaml
from ultralytics import YOLO
from IPython.display import Image, display
CONF = {'hamster': 0.45, 'head': 0.35}   # 系統實際用的信心門檻
print('請上傳 hamster_bench.zip（網站：跑分 → 資料 → 下載跑分照片）')
up = files.upload()
zipfile.ZipFile(next(n for n in up if n.endswith('.zip'))).extractall('/content')
w = pathlib.Path('/content/runs/hamster/weights/best.pt')
if not w.exists():
    print('這次沒有在這裡訓練：請上傳要測的模型 best.pt')
    w = pathlib.Path(next(iter(files.upload())))
m = YOLO(str(w))
sz = int((getattr(m, 'ckpt', None) or {}).get('train_args', {}).get('imgsz', 320) or 320)
data = '/content/hamster_bench/data.yaml'
if len(m.names) == 1:   # 舊模型只有全身：只留全身答案，避免拿不同類別數比
    shutil.copytree('/content/hamster_bench', '/content/hamster_bench_body', dirs_exist_ok=True)
    for f in pathlib.Path('/content/hamster_bench_body/labels/test').glob('*.txt'):
        f.write_text(''.join(l + '\n' for l in f.read_text().splitlines() if l.startswith('0 ')))
    y = yaml.safe_load(open(data)); y['path'] = '/content/hamster_bench_body'; y['names'] = {0: 'hamster'}
    data = '/content/hamster_bench_body/data.yaml'; yaml.safe_dump(y, open(data, 'w'), allow_unicode=True)
    print('這個模型只有全身一個類別：只算全身')

def score(model, name):
    r = model.val(data=data, split='test', imgsz=sz, project='/content/bench', name=name, exist_ok=True, plots=True, verbose=False)
    rows = {}
    for i, c in enumerate(r.box.ap_class_index):
        n = model.names[int(c)]
        rows[n] = {'AP50': r.box.ap50[i], 'AP50-95': r.box.ap[i]}
    # 固定門檻下的 P / R / F1（官方 val 的 P/R 取在 F1 最好的門檻；這裡用系統實際的門檻、IoU ≥ 0.5 一對一配對自己算）
    root = pathlib.Path(yaml.safe_load(open(data))['path'])
    imgs = sorted((root / 'images/test').iterdir())
    for n, th in CONF.items():
        if n not in rows: continue
        ci = [k for k, v in model.names.items() if v == n][0]
        tp = fp = fn = 0
        for im in imgs:
            lab = root / 'labels/test' / (im.stem + '.txt')
            g = [list(map(float, l.split()[1:])) for l in (lab.read_text().splitlines() if lab.exists() else []) if l.split() and int(l.split()[0]) == ci]
            res = model.predict(str(im), imgsz=sz, conf=th, verbose=False)[0]
            d = [(float(c), x.tolist()) for c, x, k in zip(res.boxes.conf, res.boxes.xywhn, res.boxes.cls) if int(k) == ci]
            used = set()
            for c, x in sorted(d, key=lambda t: -t[0]):
                best, bj = 0.5, None
                for j, y in enumerate(g):
                    if j in used: continue
                    ix = max(0, min(x[0] + x[2] / 2, y[0] + y[2] / 2) - max(x[0] - x[2] / 2, y[0] - y[2] / 2))
                    iy = max(0, min(x[1] + x[3] / 2, y[1] + y[3] / 2) - max(x[1] - x[3] / 2, y[1] - y[3] / 2))
                    u = x[2] * x[3] + y[2] * y[3] - ix * iy
                    if u > 0 and ix * iy / u >= best: best, bj = ix * iy / u, j
                if bj is None: fp += 1
                else: used.add(bj); tp += 1
            fn += len(g) - len(used)
        pp = tp / (tp + fp) if tp + fp else 0.0
        rr = tp / (tp + fn) if tp + fn else 0.0
        rows[n].update(P=pp, R=rr, F1=2 * pp * rr / (pp + rr) if pp + rr else 0.0, conf=th)
    return rows

before = score(m, 'pt')
print('匯出成 NCNN（樹莓派用的格式）…')
ncnn = YOLO(m.export(format='ncnn', imgsz=sz), task='detect')
after = score(ncnn, 'ncnn')
print(f'\n{"類別":8s} {"指標":8s} {"轉換前 best.pt":>14s} {"轉換後 NCNN":>12s} {"差異":>8s}')
for n in before:
    for k in ('AP50', 'AP50-95', 'P', 'R', 'F1'):
        if k in before[n]:
            a, b = before[n][k], after.get(n, {}).get(k, float('nan'))
            print(f'{n:8s} {k:8s} {a:14.3f} {b:12.3f} {b - a:+8.3f}')
print('\nP / R / F1 用系統實際的信心門檻：', {n: CONF[n] for n in before if n in CONF})
print('轉換前後的差距＝匯出／轉換造成的差異（同一模型、同一組照片）。')
for f in ('PR_curve.png', 'BoxPR_curve.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    q = pathlib.Path('/content/bench/pt') / f
    if q.exists(): display(Image(str(q), width=700))